# Data Loading: MCO vs MIA Analysis

Reads flight data from Dropbox and filters for MCO/MIA airports.

**Method:** Chunk processing - reads compressed .bz2 files directly and filters simultaneously.

## Setup

In [67]:
import pandas as pd
import numpy as np
import os

print(f"pandas {pd.__version__}")

pandas 2.1.4


## Configuration

In [68]:
# Target airports
AIRPORTS = ['MCO', 'MIA']

# Years to process
YEARS = [2004, 2005, 2006, 2007, 2008]

# Output directory
OUTPUT_DIR = '../data/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Airports: {', '.join(AIRPORTS)}")
print(f"Years: {YEARS[0]}-{YEARS[-1]}")
print(f"Output: {OUTPUT_DIR}")

Airports: MCO, MIA
Years: 2004-2008
Output: ../data/processed


## Load Dropbox URLs from secrets.txt

In [69]:
# Read Dropbox URLs from secrets.txt (not committed to GitHub)
with open('secrets.txt', 'r') as f:
    urls = [line.strip() for line in f if line.strip()]

# Map URLs to years
DROPBOX_URLS = dict(zip(YEARS, urls))

print(f"Loaded {len(DROPBOX_URLS)} URLs from secrets.txt")

Loaded 5 URLs from secrets.txt


## Load and Filter Function

In [70]:
def load_and_filter(url, year, output_dir, airports, chunksize=100000):
    """
    Load .bz2 CSV from Dropbox and filter for specified airports in chunks.
    
    Parameters:
    - url: Dropbox direct download URL
    - year: Year being processed
    - output_dir: Directory to save filtered data
    - airports: List of airport codes to filter
    - chunksize: Rows per chunk (default 100000)
    
    Returns:
    - DataFrame with filtered data
    """
    output_file = f'{output_dir}/mco_mia_{year}.csv.gz'
    
    print(f"\nProcessing {year}")
    
    chunk_list = []
    total_rows = 0
    filtered_rows = 0
    
    try:
        # Read compressed file in chunks (pandas handles .bz2 automatically)
        for i, chunk in enumerate(pd.read_csv(url, compression='bz2', chunksize=chunksize, low_memory=False), 1):
            total_rows += len(chunk)
            
            # Filter for target airports
            filtered = chunk[
                (chunk['Origin'].isin(airports)) | 
                (chunk['Dest'].isin(airports))
            ]
            
            if len(filtered) > 0:
                chunk_list.append(filtered)
                filtered_rows += len(filtered)
            
            # Progress every 10 chunks
            if i % 10 == 0:
                print(f"  Processed {total_rows:,} rows, kept {filtered_rows:,}")
        
        # Combine and save
        df = pd.concat(chunk_list, ignore_index=True)
        df.to_csv(output_file, index=False, compression='gzip')
        
        reduction = (1 - filtered_rows / total_rows) * 100
        print(f"Complete: {filtered_rows:,} / {total_rows:,} rows ({reduction:.1f}% reduction)")
        print(f"Saved to: {output_file}")
        
        return df
        
    except Exception as e:
        print(f"Error: {e}")
        return None

## Process All Years

In [71]:
for year in YEARS:
    url = DROPBOX_URLS.get(year)
    if url:
        load_and_filter(url, year, OUTPUT_DIR, AIRPORTS)
    else:
        print(f"Skipping {year}: URL not found")


Processing 2004
  Processed 1,000,000 rows, kept 43,321
  Processed 2,000,000 rows, kept 94,002
  Processed 3,000,000 rows, kept 146,376
  Processed 4,000,000 rows, kept 191,578
  Processed 5,000,000 rows, kept 237,547
  Processed 6,000,000 rows, kept 287,185
  Processed 7,000,000 rows, kept 333,392
Complete: 344,250 / 7,129,270 rows (95.2% reduction)
Saved to: ../data/processed/mco_mia_2004.csv.gz

Processing 2005
  Processed 1,000,000 rows, kept 46,709
  Processed 2,000,000 rows, kept 102,630
  Processed 3,000,000 rows, kept 153,943
  Processed 4,000,000 rows, kept 197,834
  Processed 5,000,000 rows, kept 252,802
  Processed 6,000,000 rows, kept 301,522
  Processed 7,000,000 rows, kept 350,381
Complete: 364,415 / 7,140,596 rows (94.9% reduction)
Saved to: ../data/processed/mco_mia_2005.csv.gz

Processing 2006
  Processed 1,000,000 rows, kept 50,866
  Processed 2,000,000 rows, kept 103,894
  Processed 3,000,000 rows, kept 163,280
  Processed 4,000,000 rows, kept 209,041
  Processed 5

## Combine All Years

In [72]:
dfs = []

for year in YEARS:
    filepath = f'{OUTPUT_DIR}/mco_mia_{year}.csv.gz'
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        dfs.append(df)
        print(f"{year}: {len(df):,} rows")
    else:
        print(f"{year}: file not found")

if dfs:
    # Combine all years
    df_combined = pd.concat(dfs, ignore_index=True)
    
    # Save combined file
    output_path = f'{OUTPUT_DIR}/mco_mia_2004_2008.csv.gz'
    df_combined.to_csv(output_path, index=False, compression='gzip')
    
    print(f"\nCombined: {len(df_combined):,} total rows")
    print(f"Saved to: {output_path}")
    
    # Preview
    display(df_combined.head())
    
    # Summary
    print(f"\nColumns: {len(df_combined.columns)}")
    print(f"File size: {os.path.getsize(output_path) / 1024**2:.1f} MB")
else:
    print("\nNo data files found")

2004: 344,250 rows
2005: 364,415 rows
2006: 365,595 rows
2007: 384,594 rows
2008: 133,344 rows

Combined: 1,592,198 total rows
Saved to: ../data/processed/mco_mia_2004_2008.csv.gz


,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,...,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2004,1,1,4,1622.0,1625,1955.0,2003,UA,470,...,2.0,23.0,0,NaN,0,0.0,0.0,0.0,0.0,0.0
1,2004,1,2,5,1641.0,1625,2033.0,2003,UA,470,...,2.0,47.0,0,NaN,0,0.0,0.0,14.0,0.0,16.0
2,2004,1,3,6,1644.0,1625,2000.0,2003,UA,470,...,3.0,13.0,0,NaN,0,0.0,0.0,0.0,0.0,0.0
3,2004,1,4,7,1744.0,1625,2144.0,2003,UA,470,...,4.0,48.0,0,NaN,0,12.0,0.0,22.0,0.0,67.0
4,2004,1,5,1,1637.0,1625,2006.0,2003,UA,470,...,4.0,14.0,0,NaN,0,0.0,0.0,0.0,0.0,0.0



Columns: 29
File size: 34.0 MB


## Next Step

Proceed to **02_data_cleaning.ipynb**